In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import re
import plotly.io as pio
from dateutil.relativedelta import relativedelta
import glob
import os
from calendar import monthrange
url = "https://ldcom365.sharepoint.com"
from datetime import date


# SELECTING THE MONTH OF THE REPORT

The default parameter is today's month

In [0]:
selected_date=datetime.today()
selected_month=selected_date.month
selected_year=selected_date.year

if selected_month==1:
  selected_season=f"{selected_year-1}/{selected_year}"
else:
  selected_season=f"{selected_year}/{selected_year+1}"

email_to_send=['florian.girardi-ext@ldc.com','Vitor.Sene@ldc.com'] #list of emails the report will be sent to, format: ['florian.girardi-ext@ldc.com','valentin.chiesa@ldc.com']


## Below is the code to manually select a month different from the current date

How it works:
- Change the value of the variables `selected_month` and `selected_season`.
- `selected_month` must be a number between 1 and 12.
- `selected_season` must be a string in the format `"YYYY/YYYY"`.
- Add the email addresses you want the report sent to in the list `email_to_send`. Separate multiple emails with a comma.

To execute the cell below, remove the `#` at the beginning of each row.  
You can select all the rows (CTRL+A) and press CTRL+SHIFT+7 to comment or uncomment all rows at once.

In [0]:
# selected_month=5
# selected_season="2023/2024"
# email_to_send=['florian.girardi-ext@ldc.com','Vitor.Sene@ldc.com']



# start_year, end_year = map(int, selected_season.split("/"))
# if selected_month == 1:
#     current_year = end_year
# else:
#     current_year = start_year
# selected_date = date(current_year, selected_month, 1)



## BUILDING THE REPORT

In [0]:
today = datetime.today()
today_str = today.strftime("%d.%m")

folder_path = f"/sites/grp-globalline-ups/Shared%20Documents/1.%20Global%20line-ups/South%20America/Brazil_Line%20UP_{today_str}.xlsm"

database=sp_mgr.read_pd_from_excel(folder_path) 


In [0]:

database.drop(['Quality','IMO','Berth','Waiting time', 'Shipment month','Oct-Sep marketing year', 'Calendar year','Updated','ETA','ETB','Feb-Jan marketing year'], axis=1, inplace=True)
database.rename(columns={'ETS':'Date','Destination region as per world matrix':'Region','Destination country':'Destination','MT':'Quantity'},inplace=True)
database['Date'] = pd.to_datetime(database['Date'])


In [0]:
def assign_season_corn(row):
    date = row['Date']
    if date.month > 1:  # March to December → same year
        season_start = date.year
    else:  # January, February → previous year's marketing season
        season_start = date.year - 1
    return f"{season_start}/{season_start + 1}"

# Apply to create the new column
database['Season'] = database.apply(assign_season_corn,axis=1)
# Mapping dictionary
mapping = {
    'S': 'SAILED',
    'L': 'ANNOUNCED',
    'W': 'ANNOUNCED',
    'A': 'ANNOUNCED'
}

# Replace values
database['Status'] = database['Status'].replace(mapping)


filtered_df = database[(database['Season'] == selected_season) & (database['Date'].dt.month == selected_month)]


In [0]:
# Group sailed quantities by region
sailed = filtered_df[filtered_df['Status'] == 'SAILED'].groupby('Region', as_index=False)['Quantity'].sum()
sailed = sailed.rename(columns={'Quantity': 'Quantity_Sailed'})

# Group announced quantities by region
announced = filtered_df[filtered_df['Status'] == 'ANNOUNCED'].groupby('Region', as_index=False)['Quantity'].sum()
announced = announced.rename(columns={'Quantity': 'Quantity_Announced'})

# Merge both
region_totals = pd.merge(sailed, announced, on='Region', how='outer').fillna(0)

# Add total column
region_totals['Total'] = region_totals['Quantity_Sailed'] + region_totals['Quantity_Announced']

# Optional: sort by total
region_totals = region_totals.sort_values(by='Total', ascending=False)

filtered_df_year = database[(database['Season']==selected_season)]


# Add Month column (numerical)
filtered_df_year['Month'] = filtered_df_year['Date'].dt.month

# Clean country names if needed
filtered_df_year['Destination'] = filtered_df_year['Destination'].str.upper().str.strip()

# Pivot table: Country (Destination) x Month with Quantity summed
monthly_table = (
    filtered_df_year
    .groupby(['Destination', 'Month'])['Quantity']
    .sum()
    .unstack(fill_value=0)  # Fill months with 0 if no shipments
     # Ensure months go 1 to 12
)

# Optional: round values
monthly_table = monthly_table.round(3)

# If you want to reset index and rename columns to be like 1, 2, 3...:
monthly_table.columns.name = None
monthly_table = monthly_table.reset_index()


# Clean country names if needed
filtered_df_year['Destination'] = filtered_df_year['Destination'].str.upper().str.strip()

filtered_df_year_sailed=filtered_df_year[filtered_df_year['Status']=='SAILED']
filtered_df_year_anc=filtered_df_year[filtered_df_year['Status']=='ANNOUNCED']

# Pivot table: Country (Destination) x Month with Quantity summed
monthly_table_sailed = (
    filtered_df_year_sailed
    .groupby(['Destination', 'Month'])['Quantity']
    .sum()
    .unstack(fill_value=0)  # Fill months with 0 if no shipments
         # Ensure months go 1 to 12
)

# Optional: round values
monthly_table_sailed = monthly_table_sailed.round(3)

# If you want to reset index and rename columns to be like 1, 2, 3...:
monthly_table_sailed.columns.name = None
monthly_table_sailed = monthly_table_sailed.reset_index()

# Pivot table: Country (Destination) x Month with Quantity summed
monthly_table_ancd = (
    filtered_df_year_anc
    .groupby(['Destination', 'Month'])['Quantity']
    .sum()
    .unstack(fill_value=0)  # Fill months with 0 if no shipments
   # Ensure months go 1 to 12
)

# Optional: round values
monthly_table_ancd = monthly_table_ancd.round(3)

# If you want to reset index and rename columns to be like 1, 2, 3...:
monthly_table_ancd.columns.name = None
monthly_table_ancd = monthly_table_ancd.reset_index()

from datetime import datetime

# Step 1: Add 'Month' column if missing
filtered_df_year['Month'] = filtered_df_year['Date'].dt.month

# Step 2: Get current month + 1
cutoff_month = (selected_month + 1)

# Step 3: Filter up to current month + 1
filtered_df_cut = filtered_df_year[filtered_df_year['Month'] <= cutoff_month]

# Step 4: Create pivot table by Region and Month
quantity_by_region_month = pd.pivot_table(
    filtered_df_cut,
    values='Quantity',
    index='Region',
    columns='Month',
    aggfunc='sum',
    fill_value=0
)

# Step 5: Replace month numbers with names (Jan, Feb, etc.)
quantity_by_region_month.columns = [datetime(1900, m, 1).strftime('%b') for m in quantity_by_region_month.columns]



ref_month = selected_month

# Step 3: Filter the DataFrame
database_W = database[(database['Date'].dt.month == ref_month)]

sailed_df = database_W[database_W['Status'] == 'SAILED'].copy()

# Step 2: Make sure 'Date' is a datetime column
sailed_df['Date'] = pd.to_datetime(sailed_df['Date'])

# Step 3: Sort by 'Date'
sailed_df = sailed_df.sort_values(by='Date')

# Step 4: Compute the cumulative sum of 'Quantity'
sailed_df['Sailed'] = sailed_df['Quantity'].cumsum()


# Define desired column order for marketing year starting in December
marketing_months = [2,3, 4, 5, 6, 7, 8, 9, 10, 11,12, 1]

# Function to reorder columns
def reorder_months(df):
    available_months = [m for m in marketing_months if m in df.columns]
    return df[['Destination'] + available_months]

# Apply to your DataFrames
monthly_table = reorder_months(monthly_table)
monthly_table_sailed = reorder_months(monthly_table_sailed)
monthly_table_ancd = reorder_months(monthly_table_ancd)



### EXTRACTING FORECASTS FROM MARS

In [0]:
country_mapping = {
    "S. ARABIA":"SAUDI ARABIA","SAUDI ARABI":"S. ARABIA",
    "ZZZ_Unknown":"UNKNOWN",
    "":"UNKNOWN",
    np.NaN:"UNKNOWN",
    0:"UNKNOWN",
    "Afghanistan": "Afghanistan",
    "Albania": "Albania",
    "Algeria": "Algeria",
    "Angola": "Angola",
    "Armenia": "Armenia",
    "Australia": "Australia",
    "Austria": "Austria",
    "Azerbaijan": "Azerbaijan",
    "Bahrain": "Bahrain",
    "Bangladesh": "Bangladesh",
    "Barbados": "Barbados",
    "Belarus": "Belarus",
    "Belgium&Luxembourg": "Belgium",  # Split into two countries
    "Bhutan": "Bhutan",
    "Bosnia and Herzegovina": "Bosnia and Herzegovina",
    "Botswana": "Botswana",
    "Brazil": "Brazil",
    "Brunei Darussalam": "Brunei",
    "Burkina Faso": "Burkina Faso",
    "Cambodia": "Cambodia",
    "Cameroon": "Cameroon",
    "Canada": "Canada",
    "Cape Verde": "Cabo Verde",
    "Chile": "Chile",
    "China": "China",
    "Colombia": "Colombia",
    "Congo": "Republic of the Congo",
    "Congo, the Democratic Republic of the": "Democratic Republic of Congo",
    "Cook Islands": "Cook Islands",
    "Costa Rica": "Costa Rica",
    "Croatia": "Croatia",
    "Cuba": "Cuba",
    "Cyprus": "Cyprus",
    "Czech Republic": "Czechia",
    "Côte d'Ivoire": "Ivory Coast",
    "Dominican Republic": "Dominican Republic",
    "Ecuador": "Ecuador",
    "Egypt": "Egypt",
    "El Salvador": "El Salvador",
    "Estonia": "Estonia",
    "Fiji": "Fiji",
    "Finland": "Finland",
    "France": "France",
    "French Polynesia": "French Polynesia",
    "Georgia": "Georgia",
    "Germany": "Germany",
    "Ghana": "Ghana",
    "Greece": "Greece",
    "Guatemala": "Guatemala",
    "Guinea": "Guinea",
    "Guyana": "Guyana",
    "Haiti": "Haiti",
    "Honduras": "Honduras",
    "Hong Kong": "Hong Kong",
    "Hungary": "Hungary",
    "Iceland": "Iceland",
    "India": "India",
    "Indonesia": "Indonesia",
    "Iran, Islamic Republic of": "Iran",
    "Iraq": "Iraq",
    "Ireland": "Ireland",
    "Israel": "Israel",
    "Italy": "Italy",
    "Jamaica": "Jamaica",
    "Japan": "Japan",
    "Jordan": "Jordan",
    "Kazakhstan": "Kazakhstan",
    "Korea, Republic of": "South Korea",
    "Kosovo": "Kosovo",
    "Kuwait": "Kuwait",
    "Lao People's Democratic Republic": "Laos",
    "Latvia": "Latvia",
    "Lebanon": "Lebanon",
    "Lesotho": "Lesotho",
    "Liberia": "Liberia",
    "Libya Arab Jamahiriya": "Libya",
    "Lithuania": "Lithuania",
    "Macao": "Macau",
    "Macedonia, the former Yugoslav Republic of": "North Macedonia",
    "Malaysia": "Malaysia",
    "Mali": "Mali",
    "Malta": "Malta",
    "Marshall Islands": "Marshall Islands",
    "Mauritania": "Mauritania",
    "Mauritius": "Mauritius",
    "Mexico": "Mexico",
    "Micronesia, Federated States of": "Micronesia",
    "Moldova, Republic of": "Moldova",
    "Mongolia": "Mongolia",
    "Montenegro": "Montenegro",
    "Morocco": "Morocco",
    "Mozambique": "Mozambique",
    "Myanmar": "Myanmar",
    "Namibia": "Namibia",
    "Nauru": "Nauru",
    "Nepal": "Nepal",
    "Netherlands": "Netherlands",
    "New Zealand": "New Zealand",
    "Nicaragua": "Nicaragua",
    "Niger": "Niger",
    "Nigeria": "Nigeria",
    "Norway": "Norway",
    "Oman": "Oman",
    "Pakistan": "Pakistan",
    "Panama": "Panama",
    "Papua New Guinea": "Papua New Guinea",
    "Peru": "Peru",
    "Philippines": "Philippines",
    "Poland": "Poland",
    "Portugal": "Portugal",
    "Puerto Rico": "Puerto Rico",
    "Qatar": "Qatar",
    "Romania": "Romania",
    "Russian Federation": "Russia",
    "Saint Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "Samoa": "Samoa",
    "Saudi Arabia": "Saudi Arabia",
    "Senegal": "Senegal",
    "Serbia": "Serbia",
    "Seychelles": "Seychelles",
    "Singapore": "Singapore",
    "Slovakia": "Slovakia",
    "Slovenia": "Slovenia",
    "South Africa": "South Africa",
    "Spain": "Spain",
    "Sri Lanka": "Sri Lanka",
    "Sudan": "Sudan",
    "Swaziland": "Eswatini",
    "Sweden": "Sweden",
    "Switzerland": "Switzerland",
    "Syrian Arab Republic": "Syria",
    "Taiwan, Province of China": "Taiwan",
    "Tajikistan": "Tajikistan",
    "Thailand": "Thailand",
    "Timor-Leste": "East Timor",
    "Trinidad and Tobago": "Trinidad and Tobago",
    "Tunisia": "Tunisia",
    "Turkey": "Turkey",
    "UNKNOWN": "UNKNOWN",  # Not a valid country
    "United Arab Emirates": "UAE",
    "United Kingdom": "United Kingdom",
    "United States": "United States",
    "Uruguay": "Uruguay",
    "Uzbekistan": "Uzbekistan",
    "Venezuela, Bolivarian Republic of": "Venezuela",
    "Viet Nam": "Vietnam",
    "Yemen": "Yemen",
    "Zimbabwe": "Zimbabwe",
    "WORLD": "WORLD",  # Aggregate/placeholder,
    "HOLLAND":"NETHERLANDS",
    "Holland":"Netherlands",
    "C/P":"UNKNOWN"
}


country_region_mapping = {
    "AFGHANISTAN": "ASIA",
    "ALBANIA": "EU",
    "ALGERIA": "AFRICA",
    "ANGOLA": "AFRICA",
    "ARMENIA": "ASIA",
    "AUSTRALIA": "OCEANIA",
    "AUSTRIA": "EU",
    "AZERBAIJAN": "ASIA",
    "BAHRAIN": "ME ASIA",
    "BANGLADESH": "ASIA",
    "BARBADOS": "CENTRAL AMERICA",
    "BELARUS": "ASIA",
    "BELGIUM": "EU",
    "BHUTAN": "ASIA",
    "BOSNIA AND HERZEGOVINA": "EU",
    "BOTSWANA": "AFRICA",
    "BRAZIL": "SOUTH AMERICA",
    "BRUNEI": "SE ASIA",
    "BURKINA FASO": "AFRICA",
    "CAMBODIA": "SE ASIA",
    "CAMEROON": "AFRICA",
    "CANADA": "NORTH AMERICA",
    "CABO VERDE": "AFRICA",
    "CHILE": "SOUTH AMERICA",
    "CHINA": "ASIA",
    "COLOMBIA": "SOUTH AMERICA",
    "REPUBLIC OF THE CONGO": "AFRICA",
    "DEMOCRATIC REPUBLIC OF CONGO": "AFRICA",
    "COOK ISLANDS": "OCEANIA",
    "COSTA RICA": "CENTRAL AMERICA",
    "CROATIA": "EU",
    "CUBA": "CENTRAL AMERICA",
    "CYPRUS": "EU",
    "CZECHIA": "EU",
    "IVORY COAST": "AFRICA",
    "DOMINICAN REPUBLIC": "CENTRAL AMERICA",
    "ECUADOR": "SOUTH AMERICA",
    "EGYPT": "AFRICA",
    "EL SALVADOR": "CENTRAL AMERICA",
    "ESTONIA": "EU",
    "FIJI": "OCEANIA",
    "FINLAND": "EU",
    "FRANCE": "EU",
    "FRENCH POLYNESIA": "OCEANIA",
    "GEORGIA": "ME ASIA",
    "GERMANY": "EU",
    "GHANA": "AFRICA",
    "GREECE": "EU",
    "GUATEMALA": "CENTRAL AMERICA",
    "GUINEA": "AFRICA",
    "GUYANA": "SOUTH AMERICA",
    "HAITI": "CENTRAL AMERICA",
    "HONDURAS": "CENTRAL AMERICA",
    "HONG KONG": "ASIA",
    "HUNGARY": "EU",
    "ICELAND": "EU",
    "INDIA": "ASIA",
    "INDONESIA": "SE ASIA",
    "IRAN": "ME ASIA",
    "IRAQ": "ME ASIA",
    "IRELAND": "EU",
    "ISRAEL": "ME ASIA",
    "ITALY": "EU",
    "JAMAICA": "CENTRAL AMERICA",
    "JAPAN": "ASIA",
    "JORDAN": "ME ASIA",
    "KAZAKHSTAN": "ASIA",
    "SOUTH KOREA": "ASIA",
    "KOSOVO": "EU",
    "KUWAIT": "ME ASIA",
    "LAOS": "SE ASIA",
    "LATVIA": "EU",
    "LEBANON": "ME ASIA",
    "LESOTHO": "AFRICA",
    "LIBERIA": "AFRICA",
    "LIBYA": "AFRICA",
    "LITHUANIA": "EU",
    "MACAU": "ASIA",
    "NORTH MACEDONIA": "EU",
    "MALAYSIA": "SE ASIA",
    "MALAWI": "AFRICA",
    "MALI": "AFRICA",
    "MALTA": "EU",
    "MARSHALL ISLANDS": "OCEANIA",
    "MAURITANIA": "AFRICA",
    "MAURITIUS": "AFRICA",
    "MEXICO": "CENTRAL AMERICA",
    "MICRONESIA": "OCEANIA",
    "MOLDOVA": "EU",
    "MONGOLIA": "ASIA",
    "MONTENEGRO": "EU",
    "MOROCCO": "AFRICA",
    "MOZAMBIQUE": "AFRICA",
    "MYANMAR": "SE ASIA",
    "NAMIBIA": "AFRICA",
    "NAURU": "OCEANIA",
    "NEPAL": "ASIA",
    "NETHERLANDS": "EU",
    "NEW ZEALAND": "OCEANIA",
    "NICARAGUA": "CENTRAL AMERICA",
    "NIGER": "AFRICA",
    "NIGERIA": "AFRICA",
    "NORWAY": "EU",
    "OMAN": "ME ASIA",
    "PAKISTAN": "ASIA",
    "PANAMA": "CENTRAL AMERICA",
    "PAPUA NEW GUINEA": "OCEANIA",
    "PARAGUAY": "SOUTH AMERICA",
    "PERU": "SOUTH AMERICA",
    "PHILIPPINES": "SE ASIA",
    "POLAND": "EU",
    "PORTUGAL": "EU",
    "PUERTO RICO": "CENTRAL AMERICA",
    "QATAR": "ME ASIA",
    "ROMANIA": "EU",
    "RUSSIA": "ASIA",
    "SAINT VINCENT AND THE GRENADINES": "CENTRAL AMERICA",
    "SAMOA": "OCEANIA",
    "SAUDI ARABIA": "ME ASIA",
    "SENEGAL": "AFRICA",
    "SERBIA": "EU",
    "SEYCHELLES": "AFRICA",
    "SINGAPORE": "SE ASIA",
    "SLOVAKIA": "EU",
    "SLOVENIA": "EU",
    "SOUTH AFRICA": "AFRICA",
    "SPAIN": "EU",
    "SRI LANKA": "ASIA",
    "SUDAN": "AFRICA",
    "ESWATINI": "AFRICA",
    "SWEDEN": "EU",
    "SWITZERLAND": "EU",
    "SYRIA": "ME ASIA",
    "TAIWAN": "ASIA",
    "TAJIKISTAN": "ASIA",
    "THAILAND": "SE ASIA",
    "EAST TIMOR": "SE ASIA",
    "TRINIDAD AND TOBAGO": "CENTRAL AMERICA",
    "TUNISIA": "AFRICA",
    "TURKEY": "ME ASIA",
    "UAE": "ME ASIA",
    "UNITED KINGDOM": "EU",
    "UNITED STATES": "NORTH AMERICA",
    "URUGUAY": "SOUTH AMERICA",
    "UZBEKISTAN": "ASIA",
    "VENEZUELA": "SOUTH AMERICA",
    "VIETNAM": "SE ASIA",
    "YEMEN": "ME ASIA",
    "ZIMBABWE": "AFRICA",
    "WORLD": "WORLD",
    "UNKNOWN": "UNKNOWN",
    "C/P": "UNKNOWN",
    "Unidentified":"UNKNOWN",
    
}


country_region_mapping = {
    "ALGERIA": "AFRICA",
    "ANGOLA": "AFRICA",
    "BOTSWANA": "AFRICA",
    "BURKINA FASO": "AFRICA",
    "CAMEROON": "AFRICA",
    "CAPE VERDE": "AFRICA",
    "CONGO": "AFRICA",
    "DEMOCRATIC REPUBLIC OF CONGO": "AFRICA",
    "GHANA": "AFRICA",
    "GUINEA": "AFRICA",
    "IVORY COAST": "AFRICA",
    "LESOTHO": "AFRICA",
    "LIBERIA": "AFRICA",
    "KENYA":"AFRICA",
    "UGANDA":"AFRICA",
    "MALI": "AFRICA",
    "MAURITANIA": "AFRICA",
    "MAURITIUS": "AFRICA",
    "MOROCCO": "AFRICA",
    "MOZAMBIQUE": "AFRICA",
    "NAMIBIA": "AFRICA",
    "NIGER": "AFRICA",
    "NIGERIA": "AFRICA",
    "SENEGAL": "AFRICA",
    "SEYCHELLES": "AFRICA",
    "SOUTH AFRICA": "AFRICA",
    "SUDAN": "AFRICA",
    "SWAZILAND": "AFRICA",
    "TUNISIA": "AFRICA",
    "ZIMBABWE": "AFRICA",
    "AFGHANISTAN": "ASIA",
    "BANGLADESH": "ASIA",
    "BHUTAN": "ASIA",
    "BRUNEI": "ASIA",
    "CAMBODIA": "ASIA",
    "CHINA": "ASIA",
    "HONG KONG": "ASIA",
    "INDIA": "ASIA",
    "INDONESIA": "ASIA",
    "JAPAN": "ASIA",
    "LAOS": "ASIA",
    "MACAO": "ASIA",
    "MALAYSIA": "ASIA",
    "MONGOLIA": "ASIA",
    "MYANMAR": "ASIA",
    "NEPAL": "ASIA",
    "PAKISTAN": "ASIA",
    "PHILIPPINES": "ASIA",
    "SINGAPORE": "ASIA",
    "SOUTH KOREA": "ASIA",
    "SRI LANKA": "ASIA",
    "TAIWAN": "ASIA",
    "THAILAND": "ASIA",
    "TIMOR-LESTE": "ASIA",
    "VIETNAM": "ASIA",
    "BARBADOS": "CENTRAL AMERICA",
    "COSTA RICA": "CENTRAL AMERICA",
    "CUBA": "CENTRAL AMERICA",
    "DOMINICAN REPUBLIC": "CENTRAL AMERICA",
    "EL SALVADOR": "CENTRAL AMERICA",
    "GUATEMALA": "CENTRAL AMERICA",
    "HAITI": "CENTRAL AMERICA",
    "HONDURAS": "CENTRAL AMERICA",
    "JAMAICA": "CENTRAL AMERICA",
    "NICARAGUA": "CENTRAL AMERICA",
    "PANAMA": "CENTRAL AMERICA",
    "PUERTO RICO": "CENTRAL AMERICA",
    "SAINT VINCENT": "CENTRAL AMERICA",
    "TRINIDAD AND TOBAGO": "CENTRAL AMERICA",
    "AUSTRIA": "EU",
    "BELGIUM&LUX": "EU",
    "CROATIA": "EU",
    "CYPRUS": "EU",
    "CZECH REPUBLIC": "EU",
    "ESTONIA": "EU",
    "FINLAND": "EU",
    "FRANCE": "EU",
    "GERMANY": "EU",
    "GREECE": "EU",
    "HUNGARY": "EU",
    "IRELAND": "EU",
    "ITALY": "EU",
    "LATVIA": "EU",
    "LITHUANIA": "EU",
    "MALTA": "EU",
    "NETHERLANDS": "EU",
    "POLAND": "EU",
    "PORTUGAL": "EU",
    "ROMANIA": "EU",
    "SLOVAKIA": "EU",
    "SLOVENIA": "EU",
    "SPAIN": "EU",
    "SWEDEN": "EU",
    "ALBANIA": "EUROPE NON-EU",
    "BOSNIA AND HERZEGOVINA": "EUROPE NON-EU",
    "ICELAND": "EUROPE NON-EU",
    "KOSOVO": "EUROPE NON-EU",
    "MACEDONIA": "EUROPE NON-EU",
    "MONTENEGRO": "EUROPE NON-EU",
    "NORWAY": "EUROPE NON-EU",
    "SERBIA": "EUROPE NON-EU",
    "SWITZERLAND": "EUROPE NON-EU",
    "UNITED KINGDOM": "EUROPE NON-EU",
    "ARMENIA": "FSU",
    "AZERBAIJAN": "FSU",
    "BELARUS": "FSU",
    "GEORGIA": "FSU",
    "KAZAKHSTAN": "FSU",
    "MOLDOVA": "FSU",
    "RUSSIA": "FSU",
    "TAJIKISTAN": "FSU",
    "UZBEKISTAN": "FSU",
    "BAHRAIN": "MIDDLE EAST",
    "EGYPT": "MIDDLE EAST",
    "IRAN": "MIDDLE EAST",
    "IRAQ": "MIDDLE EAST",
    "ISRAEL": "MIDDLE EAST",
    "JORDAN": "MIDDLE EAST",
    "KUWAIT": "MIDDLE EAST",
    "LEBANON": "MIDDLE EAST",
    "LIBYA": "MIDDLE EAST",
    "OMAN": "MIDDLE EAST",
    "QATAR": "MIDDLE EAST",
    "SAUDI ARABIA": "MIDDLE EAST",
    "SYRIA": "MIDDLE EAST",
    "TURKEY": "MIDDLE EAST",
    "UAE": "MIDDLE EAST",
    "YEMEN": "MIDDLE EAST",
    "CANADA": "NORTH AMERICA",
    "MEXICO": "NORTH AMERICA",
    "UNITED STATES": "NORTH AMERICA",
    "AUSTRALIA": "OCEANIA",
    "COOK ISLANDS": "OCEANIA",
    "FIJI": "OCEANIA",
    "FRENCH POLYNESIA": "OCEANIA",
    "MARSHALL ISLANDS": "OCEANIA",
    "MICRONESIA": "OCEANIA",
    "NAURU": "OCEANIA",
    "NEW ZEALAND": "OCEANIA",
    "PAPUA NEW GUINEA": "OCEANIA",
    "SAMOA": "OCEANIA",
    "BRAZIL": "SOUTH AMERICA",
    "CHILE": "SOUTH AMERICA",
    "COLOMBIA": "SOUTH AMERICA",
    "ECUADOR": "SOUTH AMERICA",
    "GUYANA": "SOUTH AMERICA",
    "PERU": "SOUTH AMERICA",
    "URUGUAY": "SOUTH AMERICA",
    "VENEZUELA": "SOUTH AMERICA",
    "WORLD": "WORLD",
    "UNKNOWN": "UNKNOWN",
    "C/P": "UNKNOWN",
    "Unidentified":"UNKNOWN"

}



In [0]:
# Initialize allowing me to access Sharepoint
databricks_init(dbutils, "GO")


db = SqlManager()


# {"at price", "residual"}
# TYPE = "At Price"
TYPE = "At price"
# TYPE = "Residual"

#### Constants

# Countries to graph
want_countries = {"Brazil"}#, "United States"}
cropyearstart = {'United States': 9, 'Ukraine': 10, 'Brazil': 2, 'Argentina': 3}


## Colors

# LDC colors
navy = "#32556E"
green = "#599536"
lightblue = "#97B8DB"
purple = "#5D465C"
lowablue = "#b8c1ff"




# Sets of colors
# [pale, saturated, dark]
redset = ["#ffadad", "#7e0000", "#ff0000"]
blueset = ["#c7cfff", "#000e65", "#0023ff"]
greenset = ["#a2cc95", "#1a531e", "#00b712"]
brownset = ["#ffddb9", "#3ca947", "#765a2d"] 

colorset = [redset, blueset, greenset, brownset]


# Set matrix to look for at-price vs residual
exp_val = ["At-price", True]
if (TYPE.title() == "Residual".title()):
  exp_val[0] = "Expected"


#### Read in from sql database

tradeflow_read = db.sql_read('MarsGrainsReplica', 
                        table = 'dbo.TradeFlow')


#### Sql keys to codes

country_id = db.sql_read('MarsGrainsReplica',table = 'dbo.Country')
commodity_id = db.sql_read('MarsGrainsReplica', table = 'dbo.Commodity')
commodity_quality_id = db.sql_read('MarsGrainsReplica', table = 'dbo.CommodityQuality')
commodity_sub_quality_id = db.sql_read('MarsGrainsReplica', table = 'dbo.CommoditySubQuality')
status_id = db.sql_read('MarsGrainsReplica', table = 'dbo.Status')


#### Rename key dfs to match the bs read in

# Country
country_id["Country"] = country_id["Name"]
country_id["country_id"] = country_id["Id"]

# Commodity
commodity_id["Commodity"] = commodity_id["Name"]
commodity_id["commodity_id"] = commodity_id["Id"]

# Quality
commodity_quality_id["CommodityQuality"] = commodity_quality_id["Name"]
commodity_quality_id["quality_id"] = commodity_quality_id["Id"]
commodity_quality_id["commodity_id"] = commodity_quality_id["Commodity"]

# Subquality
commodity_sub_quality_id["CommoditySubQuality"] = commodity_sub_quality_id["Name"]
commodity_sub_quality_id["subquality_id"] = commodity_sub_quality_id["Id"]
commodity_sub_quality_id["quality_id"] = commodity_sub_quality_id["CommodityQuality"]

# Get rid of extra columns to prevent column name problems
country_id = country_id[["country_id", "Country"]]
commodity_id = commodity_id[["Commodity", "commodity_id"]]
commodity_quality_id = commodity_quality_id[["commodity_id", "quality_id", "CommodityQuality"]]
commodity_sub_quality_id = commodity_sub_quality_id[["CommoditySubQuality", "subquality_id", "quality_id"]]


#### Map IDs

#### Get rid of unmapped countries

# Init
tradeflow = tradeflow_read.copy(deep = True)

# Rename b/s df to match the ids
tradeflow = tradeflow.rename({"SourceCountry": "country_id",            
                "Commodity": "commodity_id",
                "Quality": "quality_id",
                "SubQuality": "subquality_id"}, 
                axis = 'columns')
                

## Merge codes into main df

# Country
tradeflow = pd.merge_ordered(tradeflow, country_id, left_on = ["country_id"], right_on = ["country_id"])

# Shuffle names for columns to get the destination country as well
tradeflow = tradeflow.rename({"country_id": "source_country_id", 
                              "Country": "SourceCountry",
                              "DestinationCountry": "country_id"},
                              axis = 'columns')
tradeflow = pd.merge_ordered(tradeflow, country_id, left_on = ["country_id"], right_on = ["country_id"])

# Commodity
tradeflow = pd.merge_ordered(tradeflow, commodity_id, left_on = ["commodity_id"], right_on = ["commodity_id"])
tradeflow = pd.merge_ordered(tradeflow, commodity_quality_id, left_on = ["quality_id", "commodity_id"], right_on = ["quality_id", "commodity_id"])
tradeflow = pd.merge_ordered(tradeflow, commodity_sub_quality_id, left_on = ["subquality_id", "quality_id"], right_on = ["subquality_id", "quality_id"])

# Rename to match original tradeflows name
tradeflow = tradeflow.rename({"CommodityQuality": "Quality", 
                              "CommoditySubQuality": "SubQuality",
                              "Country": "DestinationCountry",
                              "Status": "status_id"},
                              axis = 'columns')


#### Get rid of unmapped countries



# Get rid of extras
tradeflow = tradeflow.loc[tradeflow["Commodity"] == "Corn"]
tradeflow = tradeflow.loc[tradeflow["Quality"] == "All Corn"]
tradeflow = tradeflow.loc[tradeflow["SubQuality"] == "Yellow Corn"]


# # Get rid of null entries
tradeflow = tradeflow.loc[tradeflow["country_id"].notna()]

# # Keep only countries on my list
tradeflow = tradeflow.loc[tradeflow["SourceCountry"].isin(want_countries)]

#### Map status

# Rename status matrix
status_id = status_id.rename({"Id": "status_id",
                  "Name": "Status"}, 
                 axis = 1)

# Merge
tradeflow = pd.merge_ordered(tradeflow, status_id, left_on = ["status_id"], right_on = ["status_id"])

#### Cleanup


# Keep only shipped and at-price
# tradeflow = tradeflow.loc[tradeflow["Status"].isin(["Shipped", "Expected"])]


# Helper column for market year
tradeflow["my_start"] = tradeflow["SourceCountry"].map(cropyearstart)
tradeflow["newcrop"] = tradeflow["Month"] >= tradeflow["my_start"]

# Get market year
tradeflow["my"] = tradeflow["Year"]
tradeflow["my"].loc[~tradeflow["newcrop"]] = tradeflow["my"] - 1

# Select columns
tradeflow = tradeflow[["SourceCountry",  "DestinationCountry", "Year", "Month", "Value", "my", "newcrop", "my_start", "Status", "IsExporter"]]

tradeflow['Date']= pd.to_datetime(dict(year=tradeflow['Year'], month=tradeflow['Month'], day=1))

# Apply to create the new column
tradeflow['Season'] = tradeflow.apply(assign_season_corn, axis=1)

current_season=selected_season

tradeflow_current_season=tradeflow.loc[tradeflow['Season'] == current_season]
tradeflow_current_season=tradeflow_current_season.drop(columns=['SourceCountry','my','my_start','Season','newcrop','IsExporter','Year'])
tradeflow_current_season["DestinationCountry"] = tradeflow_current_season["DestinationCountry"].replace('ZZZ_Unknown_Destination', 'UNKNOWN')
filtered_tradeflow_current_season = tradeflow_current_season[tradeflow_current_season['Status'].isin(['At-price', 'Shipped'])]

filtered_tradeflow_current_season.drop(columns=['Status'])



tradeflow_current_season_pivot = filtered_tradeflow_current_season.pivot_table(
    index='DestinationCountry',
    columns='Month',
    values='Value',
    aggfunc='sum'  # or 'mean', 'first', etc. depending on your need
)

tradeflow_current_season_pivot.columns.name = None
tradeflow_current_season_pivot.columns = tradeflow_current_season_pivot.columns.astype(int)

# Reorder columns: March to December, then January and February
ordered_cols = list(range(2, 13)) + [1]
tradeflow_current_season_pivot = tradeflow_current_season_pivot[ordered_cols]

world_row = tradeflow_current_season_pivot.sum(numeric_only=True)

# Assign it to a new row named 'WORLD'
tradeflow_current_season_pivot.loc['WORLD'] = world_row



tradeflow_current_season_pivot.reset_index(inplace=True)
tradeflow_current_season_pivot['DestinationCountry'] = tradeflow_current_season_pivot['DestinationCountry'].map(country_mapping)
tradeflow_current_season_pivot.rename(columns={'DestinationCountry': '0'}, inplace=True)
pivot_df=tradeflow_current_season_pivot.copy()


In [0]:


# Rename first column to 'Country'
df1 = monthly_table.rename(columns={monthly_table.columns[0]: "Country"})
df2 = pivot_df.rename(columns={pivot_df.columns[0]: "Country"})

# Normalize country names to uppercase (or lowercase, your choice)
df1["Country"] = df1["Country"].str.upper()
df2["Country"] = df2["Country"].str.upper()


# Extract the month (as an integer)
month_col =selected_month


# Extract and rename relevant columns
lineups = df1[["Country", month_col]].rename(columns={month_col: "Lineup"})
forecasts = df2[["Country", month_col]].rename(columns={month_col: "Forecast"})

# Merge
merged = pd.merge(lineups, forecasts, on="Country", how="outer")

# Optional: sort
merged = merged.sort_values("Country").reset_index(drop=True)

merged['Forecast']=merged['Forecast']*1000
merged['Lineup']=merged['Lineup'].round()

merged['Country'] = merged['Country'].apply(lambda x: country_mapping.get(x, x))
merged['Region'] = merged['Country'].map(country_region_mapping)
merged['Var']=merged['Lineup']-merged['Forecast']

# Group by 'Region' and calculate the sum for each region
region_subtotals = merged.groupby('Region').agg({
    'Lineup': 'sum',
    'Forecast': 'sum',
    'Var': 'sum'
}).reset_index()

# Add a column for the subtotal row name
region_subtotals['Country'] = 'Subtotal'

# Append the subtotal rows to the merged dataframe
merged_with_subtotals = pd.concat([merged, region_subtotals], ignore_index=True)

# Sort the dataframe to place subtotal rows at the end of each region
merged_with_subtotals['Region_Order'] = merged_with_subtotals['Region'].map({
    'AFRICA': 0, 'ASIA': 1, 'CENTRAL AMERICA': 2, 'EU': 3,
    'EUROPE NON-EU': 4,'FSU':5, 'MIDDLE EAST': 6, 'NORTH AMERICA': 7,'OCEANIA': 8,'SOUTH AMERICA': 9,'UNKNOWN':10,"WORLD":11
})

merged_with_subtotals = merged_with_subtotals.sort_values(by=['Region_Order', 'Country'])

# Drop the 'Region_Order' column
merged_with_subtotals = merged_with_subtotals.drop(columns=['Region_Order'])


region_order = ['AFRICA', 'ASIA', 'CENTRAL AMERICA', 'EU','EUROPE NON-EU','FSU', 'MIDDLE EAST', 'NORTH AMERICA','OCEANIA','SOUTH AMERICA','UNKNOWN',"WORLD"]

# Ensure Region is a categorical column with order
merged_with_subtotals['Region'] = pd.Categorical(
    merged_with_subtotals['Region'],
    categories=region_order,
    ordered=True
)

# Replace 'Subtotal' country entries with 'Subtotal [Region]'
merged_with_subtotals.loc[
    merged_with_subtotals['Country'] == 'Subtotal',
    'Country'
] = 'Subtotal ' + merged_with_subtotals['Region'].astype(str)

# Sort values by Region and then within Region put Subtotal at the end
def custom_sort(df):
    # Put all non-subtotals first, then the subtotal
    subtotals = df[df['Country'].str.startswith('Subtotal')]
    others = df[~df['Country'].str.startswith('Subtotal')]
    return pd.concat([others, subtotals])

# Apply the sorting logic by region
merged_with_subtotals = (
    merged_with_subtotals
    .sort_values(['Region', 'Country'])  # Preliminary sort
    .groupby('Region', group_keys=False)
    .apply(custom_sort)
)
merged_with_subtotals = merged_with_subtotals[merged_with_subtotals['Country'] != 'Subtotal UNKNOWN']
merged_with_subtotals = merged_with_subtotals[merged_with_subtotals['Country'] != 'Subtotal WORLD']

# Optionaleset index or keep Region as index
merged_with_subtotals.set_index(['Region', 'Country'], inplace=True)

total_lineup = merged_with_subtotals[~merged_with_subtotals.index.get_level_values('Country').str.contains('Subtotal')]['Lineup'].sum()

# Set the value for the WORLD row
merged_with_subtotals.loc[('WORLD', 'WORLD'), 'Lineup'] = total_lineup


merged_with_subtotals = merged_with_subtotals.rename(columns={'Forecast': 'BS'})

# Now set the 'Var' column for the 'WORLD' row
merged_with_subtotals.loc[('WORLD', 'WORLD'), 'Var'] = merged_with_subtotals.loc[('WORLD', 'WORLD'), 'Lineup'] - merged_with_subtotals.loc[('WORLD', 'WORLD'), 'BS']


merged_with_subtotals_ht=merged_with_subtotals.fillna(0)

# Drop rows where both Lineup and Forecast are zero
merged_with_subtotals_ht= merged_with_subtotals_ht[~((merged_with_subtotals_ht['Lineup'] == 0) & (merged_with_subtotals_ht['BS'] == 0))]

# Format numeric values with thousand separators and no decimals
merged_with_subtotals_ht[['Lineup', 'BS', 'Var']] = merged_with_subtotals_ht[['Lineup', 'BS', 'Var']].applymap(lambda x: f"{x/1000:,.0f}k")

merged_with_subtotals_ht.rename(columns={'BS':'At Price'},inplace=True)

total_lineup = merged_with_subtotals[~merged_with_subtotals.index.get_level_values('Country').str.contains('Subtotal')]['Lineup'].sum()




In [0]:
pivot_df=tradeflow_current_season_pivot.copy()

# Replace values in columns
pivot_df["0"] = pivot_df["0"].replace('ZZZ_Unknown', 'UNKNOWN')
pivot_df["0"] = pivot_df["0"].replace('World: total areas', 'WORLD')

# Rename first column to 'Country'
df1 = monthly_table.rename(columns={monthly_table.columns[0]: "Country"})
df2 = pivot_df.rename(columns={pivot_df.columns[0]: "Country"})

# Normalize country names to uppercase (or lowercase, your choice)
df1["Country"] = df1["Country"].str.upper()
df2["Country"] = df2["Country"].str.upper()

# Extract the month (as an integer)
if selected_month<12:
    month_col=selected_month+1
else:
    month_col =1


if month_col in df1.columns and month_col in df2.columns:
    lineups = df1[["Country", month_col]].rename(columns={month_col: "Lineup"})
    forecasts = df2[["Country", month_col]].rename(columns={month_col: "Forecast"})
else:
    # Create empty placeholders if data is missing
    lineups = df1[["Country"]].copy()
    lineups["Lineup"] = 0

    forecasts = df2[["Country"]].copy()
    forecasts["Forecast"] = 0
    
# Merge
merged_next = pd.merge(lineups, forecasts, on="Country", how="outer")

# Optional: sort
merged_next = merged_next.sort_values("Country").reset_index(drop=True)

merged_next['Forecast']=merged_next['Forecast']*1000
merged_next['Lineup']=merged_next['Lineup'].round()

merged_next['Country'] = merged_next['Country'].apply(lambda x: country_mapping.get(x, x))
merged_next['Region'] = merged_next['Country'].map(country_region_mapping)
merged_next['Var']=merged_next['Lineup']-merged_next['Forecast']

# Group by 'Region' and calculate the sum for each region
region_subtotals = merged_next.groupby('Region').agg({
    'Lineup': 'sum',
    'Forecast': 'sum',
    'Var': 'sum'
}).reset_index()

# Add a column for the subtotal row name
region_subtotals['Country'] = 'Subtotal'

# Append the subtotal rows to the merged_next dataframe
merged_with_subtotals_next = pd.concat([merged_next, region_subtotals], ignore_index=True)

# Sort the dataframe to place subtotal rows at the end of each region
merged_with_subtotals_next['Region_Order'] = merged_with_subtotals_next['Region'].map({
    'AFRICA': 0, 'ASIA': 1, 'CENTRAL AMERICA': 2, 'EU': 3,
    'EUROPE NON-EU': 4,'FSU':5, 'MIDDLE EAST': 6, 'NORTH AMERICA': 7,'OCEANIA': 8,'SOUTH AMERICA': 9,'UNKNOWN':10,"WORLD":11
})

merged_with_subtotals_next = merged_with_subtotals_next.sort_values(by=['Region_Order', 'Country'])

# Drop the 'Region_Order' column
merged_with_subtotals_next = merged_with_subtotals_next.drop(columns=['Region_Order'])


region_order = ['AFRICA', 'ASIA', 'CENTRAL AMERICA', 'EU','EUROPE NON-EU','FSU', 'MIDDLE EAST', 'NORTH AMERICA','OCEANIA','SOUTH AMERICA','UNKNOWN',"WORLD"]

# Ensure Region is a categorical column with order
merged_with_subtotals_next['Region'] = pd.Categorical(
    merged_with_subtotals_next['Region'],
    categories=region_order,
    ordered=True
)

# Replace 'Subtotal' country entries with 'Subtotal [Region]'
merged_with_subtotals_next.loc[
    merged_with_subtotals_next['Country'] == 'Subtotal',
    'Country'
] = 'Subtotal ' + merged_with_subtotals_next['Region'].astype(str)

# Sort values by Region and then within Region put Subtotal at the end
def custom_sort(df):
    # Put all non-subtotals first, then the subtotal
    subtotals = df[df['Country'].str.startswith('Subtotal')]
    others = df[~df['Country'].str.startswith('Subtotal')]
    return pd.concat([others, subtotals])

# Apply the sorting logic by region
merged_with_subtotals_next = (
    merged_with_subtotals_next
    .sort_values(['Region', 'Country'])  # Preliminary sort
    .groupby('Region', group_keys=False)
    .apply(custom_sort)
)
merged_with_subtotals_next = merged_with_subtotals_next[merged_with_subtotals_next['Country'] != 'Subtotal UNKNOWN']
merged_with_subtotals_next = merged_with_subtotals_next[merged_with_subtotals_next['Country'] != 'Subtotal WORLD']

# Optionaleset index or keep Region as index
merged_with_subtotals_next.set_index(['Region', 'Country'], inplace=True)

total_lineup = merged_with_subtotals_next[~merged_with_subtotals_next.index.get_level_values('Country').str.contains('Subtotal')]['Lineup'].sum()

# Set the value for the WORLD row
merged_with_subtotals_next.loc[('WORLD', 'WORLD'), 'Lineup'] = total_lineup


merged_with_subtotals_next = merged_with_subtotals_next.rename(columns={'Forecast': 'BS'})

# Now set the 'Var' column for the 'WORLD' row
merged_with_subtotals_next.loc[('WORLD', 'WORLD'), 'Var'] = merged_with_subtotals_next.loc[('WORLD', 'WORLD'), 'Lineup'] - merged_with_subtotals_next.loc[('WORLD', 'WORLD'), 'BS']


merged_with_subtotals_ht_next=merged_with_subtotals_next.fillna(0)

# Drop rows where both Lineup and Forecast are zero
merged_with_subtotals_ht_next= merged_with_subtotals_ht_next[~((merged_with_subtotals_ht_next['Lineup'] == 0) & (merged_with_subtotals_ht_next['BS'] == 0))]

# Format numeric values with thousand separators and no decimals
merged_with_subtotals_ht_next[['Lineup', 'BS', 'Var']] = merged_with_subtotals_ht_next[['Lineup', 'BS', 'Var']].applymap(lambda x: f"{x/1000:,.0f}k")

merged_with_subtotals_ht_next.rename(columns={'BS':'At Price'},inplace=True)





In [0]:

merged_with_subtotals_html=merged_with_subtotals_ht.to_html()
merged_with_subtotals_html_next=merged_with_subtotals_ht_next.to_html()

## CHART OVERVIEW

In [0]:
# Extract the month (as an integer)
month_col = selected_month

BS_world = merged_with_subtotals.reset_index()
BS_value = BS_world.loc[BS_world['Country'] == 'WORLD', 'BS'].values[0]

sailed_df = filtered_df[filtered_df['Status'] == 'SAILED'].copy()

# Step 2: Make sure 'Date' is a datetime column
sailed_df['Date'] = pd.to_datetime(sailed_df['Date'])

# Step 3: Sort by 'Date'
sailed_df = sailed_df.sort_values(by='Date')

# Step 4: Compute the cumulative sum of 'Quantity'
sailed_df['Sailed'] = sailed_df['Quantity'].cumsum()

# Step 1: Filter for Status being 'SAILED' or 'ANNOUNCED'
forecast_df = filtered_df[filtered_df['Status'].isin(['SAILED', 'ANNOUNCED'])].copy()

# Step 2: Make sure 'Date' is datetime
forecast_df['Date'] = pd.to_datetime(forecast_df['Date'])

# Step 3: Sort by date
forecast_df = forecast_df.sort_values(by='Date')

# Step 4: Compute cumulative sum of Quantity
forecast_df['Forecasts'] = forecast_df['Quantity'].cumsum()

daily_totals_sailed = sailed_df.groupby('Date', as_index=False)['Quantity'].sum()

# Step 2: Compute the cumulative sum
daily_totals_sailed['Sailed'] = daily_totals_sailed['Quantity'].cumsum()

last_sailed=daily_totals_sailed.iloc[-1]
daily_totals_forecast = forecast_df.groupby('Date', as_index=False)['Quantity'].sum()

# Step 2: Compute the cumulative sum
daily_totals_forecast['Forecasts'] = daily_totals_forecast['Quantity'].cumsum()
daily_totals_forecast['Date'] = pd.to_datetime(daily_totals_forecast['Date']).dt.date

# Split season into years
start_year, end_year = map(int, selected_season.split("/"))

# If January → use second year of the season, else → use first year
if selected_month == 1:
    year = end_year
else:
    year = start_year

# Number of days in that month
num_days = monthrange(year, selected_month)[1]

# Create date range
start_date = datetime(year, selected_month, 1)
end_date = datetime(year, selected_month, num_days)
all_dates = pd.date_range(start=start_date, end=end_date, freq='D').date

# Ensure forecast date has no time
daily_totals_forecast['Date'] = pd.to_datetime(daily_totals_forecast['Date']).dt.date

# Create DataFrame of full month dates
full_dates_df = pd.DataFrame({'Date': all_dates})

# Merge to preserve values and fill missing dates
daily_totals_forecast = full_dates_df.merge(
    daily_totals_forecast, on='Date', how='left'
)

# Fill missing Quantity values
daily_totals_forecast['Quantity'] = daily_totals_forecast['Quantity'].fillna(0)

# Recalculate cumulative Forecasts
daily_totals_forecast['Forecasts'] = daily_totals_forecast['Quantity'].cumsum()


daily_totals_forecast['BS_per_day'] = BS_value / num_days
daily_totals_forecast['BS'] = daily_totals_forecast['BS_per_day'].cumsum()

last_forecast = daily_totals_forecast.iloc[-1]

In [0]:
# 1. Get the latest sailed date
last_date = daily_totals_sailed['Date'].max()

# 2. Filter the last few days for pace calculation (e.g. last 3 days)
recent_data = daily_totals_sailed[daily_totals_sailed['Date'] >= last_date - pd.Timedelta(days=5)]
recent_data = recent_data.sort_values('Date')

# 3. Compute daily increases
recent_data['Delta'] = recent_data['Sailed'].diff()
daily_pace = recent_data['Delta'].mean()

# 4. Days remaining in the month
end_of_month = last_date.replace(day=1) + pd.offsets.MonthEnd(0)
days_remaining = (end_of_month - last_date).days

last_sailed_value = recent_data['Sailed'].iloc[-1]
projected_total = last_sailed_value + daily_pace * days_remaining

# 6. Add a dashed projection line
projection_x = [last_date, end_of_month]
projection_y = [last_sailed_value, projected_total]

projected_total = 0 if pd.isna(projected_total) else projected_total
text = f"{int(projected_total/1000):,} kmt (proj.)"

In [0]:

overview = go.Figure()

# Sailed line
overview.add_trace(go.Scatter(
    x=daily_totals_sailed['Date'],
    y=daily_totals_sailed['Sailed'],
    mode='lines+markers',
    name='Sailed',
    line=dict(color='blue')
))

# Forecast line
overview.add_trace(go.Scatter(
    x=daily_totals_forecast['Date'],
    y=daily_totals_forecast['Forecasts'],
    mode='lines+markers',
    name='Line up (sailed+announced)',
    line=dict(color='orange', dash='dash')
))

# Forecast line
overview.add_trace(go.Scatter(
    x=daily_totals_forecast['Date'],
    y=daily_totals_forecast['BS'],
    mode='lines+markers',
    name='At Price exports',
    line=dict(color='red', dash='dashdot')
))


# Add annotations
overview.add_annotation(
    x=last_sailed['Date'],
    y=last_sailed['Sailed'],
    text=f"{int(last_sailed['Sailed']/1000):,.1f} kmt",
    showarrow=True,
    arrowhead=1,
    ax=60,
    ay=60,
    font=dict(color="blue")
)

overview.add_annotation(
    x=last_forecast['Date'],
    y=last_forecast['Forecasts'],
    text=f"{int(last_forecast['Forecasts']/1000):,.1f} kmt",
    showarrow=True,
    arrowhead=1,
    ax=70,
    ay=-5,
    font=dict(color="orange")
)

overview.add_annotation(
    x=last_forecast['Date'],
    y=last_forecast['BS'],
    text=f"{int(last_forecast['BS']/1000):,.1f} kmt",
    showarrow=True,
    arrowhead=1,
    ax=70,
    ay=-20,
    font=dict(color="red")
)


overview.add_trace(go.Scatter(
    x=projection_x,
    y=projection_y,
    mode='lines+markers',
    name='Projected Sailed',
    line=dict(color='blue', dash='dot')
))

# 7. Add annotation at end of projection
overview.add_annotation(
    x=end_of_month,
    y=projected_total,
    text=f"{int(projected_total/1000):,} kmt (proj.)",
    showarrow=True,
    arrowhead=1,
    ax=40,
    ay=80,
    font=dict(color="blue")
)

# Layout
overview.update_layout(
    title='Sailed vs Forecasts',
    xaxis_title='Date',
    yaxis_title='Quantity',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99),
    height=700,
    width=1200,
    hovermode='x unified'
)

# overview.add_trace(go.Scatter(
#     x=[last_week_date],
#     y=[last_week_sailed],
#     mode='markers',
#     name='Last Week level',
#     marker=dict(color='green', size=10, symbol='x')
# ))

# # Annotate the point
# overview.add_annotation(
#     x=last_week_date,
#     y=last_week_sailed,
#     text="Last Week level",
#     showarrow=True,
#     arrowhead=1,
#     ax=20,
#     ay=80,
#     font=dict(color="green")
# )
overview.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:,.0f} mt<extra>%{fullData.name}</extra>')


# Export to image
pio.write_image(overview, "sailed_vs_forecast.png", width=1000, height=600)

overview.show()


overview_html = overview.to_html(include_plotlyjs='cdn', full_html=True)

###SUMMARY

In [0]:
filtered_df
sailed_sum = filtered_df[filtered_df['Status'] == 'SAILED']['Quantity'].sum()
announced_sum = filtered_df[filtered_df['Status'] == 'ANNOUNCED']['Quantity'].sum()
total=sailed_sum+announced_sum 
forecast_BS=last_forecast['BS']
percent_vs_mars=sailed_sum/forecast_BS*100

summary_df = pd.DataFrame([{
    'Sailed': f"{round(sailed_sum / 1000)} kmt",
    'Announced': f"{round(announced_sum / 1000)} kmt",
    'Total': f"{round(total / 1000)} kmt",
    'MARS': f"{round(forecast_BS / 1000) }kmt",
    '% vs MARS': round(percent_vs_mars)
}])


In [0]:
summary_df_html=summary_df.to_html(classes='table table-striped table-bordered', index=False)

In [0]:


month_col = selected_month


# Ensure month column exists, otherwise fill with zeros
anncd = monthly_table_ancd[["Destination"]].copy()
anncd["Announced"] = monthly_table_ancd.get(month_col, pd.Series([0]*len(monthly_table_ancd)))

sailed = monthly_table_sailed[["Destination"]].copy()
sailed["Sailed"] = monthly_table_sailed.get(month_col, pd.Series([0]*len(monthly_table_sailed)))

anncd.rename(columns={'Destination': 'Country'}, inplace=True)
sailed.rename(columns={'Destination': 'Country'}, inplace=True)

anncd['Country'] = anncd['Country'].apply(lambda x: country_mapping.get(x, x))
sailed['Country'] = sailed['Country'].apply(lambda x: country_mapping.get(x, x))

merged_bar=pd.merge(anncd, sailed, on='Country',how='outer')

merged_bar=pd.merge(merged, merged_bar, on='Country', how='outer')
merged_bar=merged_bar.sort_values('Region')

excluded_destinations = ['SOUTH AMERICA', 'UNKNOWN', 'WORLD', 'AFRICA', 'ZZZ_UNKNOWN_DESTINATION']
merged_bar = merged_bar[~merged_bar['Country'].isin(excluded_destinations)]

# Filter out rows where Sailed, Announced, and Forecast are all 0 or NaN
merged_bar = merged_bar[~(((merged_bar['Sailed'].fillna(0) == 0) & (merged_bar['Announced'].fillna(0) == 0) & (merged_bar['Forecast'].fillna(0) == 0)))]

merged_bar['Total'] = merged_bar['Sailed'].fillna(0) + merged_bar['Announced'].fillna(0)

# Sort by this total and keep only top 20
merged_bar = merged_bar.sort_values('Total', ascending=False).head(20)

# Optional: sort again for cleaner plotting (descending)
merged_bar = merged_bar.sort_values('Total', ascending=True) 

# Step 1: Create the base chart with Sailed + Announced side by side
bar = go.Figure()

# Sailed
bar.add_trace(go.Bar(
    y=merged_bar['Country'],
    x=merged_bar['Sailed'],
    orientation='h',
    name='Sailed',
    marker=dict(color='steelblue'),
    offsetgroup=0,
    base=0,
    text=merged_bar['Sailed'].apply(lambda x: f"{x:,.0f}"),  # Adding text to bars
    textposition='inside',  # Position text inside the bar
    textfont=dict(size=30)  
))

# Announced
bar.add_trace(go.Bar(
    y=merged_bar['Country'],
    x=merged_bar['Announced'],
    orientation='h',
    name='Announced',
    marker=dict(color='orange'),
    offsetgroup=0,
    base=merged_bar['Sailed'],
    text=merged_bar['Announced'].apply(lambda x: f"{x:,.0f}"),  # Adding text to bars
    textposition='inside',  # Position text inside the bar
    textfont=dict(size=30)  
))

# Step 2: Add Forecast with overlay
bar.add_trace(go.Bar(
    y=merged_bar['Country'],
    x=merged_bar['Forecast'],
    orientation='h',
    name='At Price Forecasts',
    marker=dict(color='rgba(255, 0, 0, 0.4)'),  # Transparent red
    offsetgroup=1,
    base=0,
    opacity=0.4,
    text=merged_bar['Forecast'].apply(lambda x: f"{x:,.0f}"),  # Adding text to bars
    textposition='inside',  # Position text inside the bar
    textfont=dict(size=60),
))

# Layout
bar.update_layout(
    barmode='group',  # This allows Forecast to overlap
    title='Sailed and Announced vs Balance Sheet',
    xaxis_title='Quantity (mt)',
    yaxis_title='Country',
    template='plotly_white',
    height=900,
    width=900,
)

bar.show()

bar_html = bar.to_html(include_plotlyjs='cdn', full_html=True)

## TOP 10

In [0]:
# Add new column for the total of other columns
monthly_table['Total'] = monthly_table.iloc[:, 1:].sum(axis=1)

# Step 2: Select top 10 countries with the highest total
top_10_countries = monthly_table.nlargest(10, 'Total')

marketing_month_order = [2,3, 4, 5, 6, 7, 8, 9, 10, 11,12,1]
marketing_month_names = ['Feb','Mar', 'Apr', 'May', 'Jun',
                         'Jul', 'Aug', 'Sep', 'Oct', 'Nov','Dec', 'Jan']
# Step 4: Determine which month columns are currently in the DataFrame (as numbers)
month_cols = [col for col in marketing_month_order if col in monthly_table.columns]

# Step 5: Create the new column labels (starting with 'Destination' and ending with 'Total')
new_columns = ['Destination'] + [marketing_month_names[marketing_month_order.index(m)] for m in month_cols] + ['Total']

# Step 6: Apply new column names to top_10_countries
top_10_countries.columns = new_columns

# Add Total column
monthly_table['Total'] = monthly_table.iloc[:, 1:].sum(axis=1)

# Select top 10 countries
top_10_countries = monthly_table.nlargest(10, 'Total')

month_cols = [m for m in marketing_month_order if m in monthly_table.columns]
top_10_countries = top_10_countries[['Destination'] + month_cols + ['Total']]
top_10_countries.columns = ['Destination'] + [marketing_month_names[marketing_month_order.index(m)] for m in month_cols] + ['Total']


In [0]:
# Step 1: Determine month labels based on columns (excluding 'Destination' and 'Total')
month_labels = new_columns[1:-1]  # Already ordered and renamed in previous steps

# Step 2: Create the plot
top = go.Figure()

# Step 3: Add a trace for each of the top 10 countries
for idx, row in top_10_countries.iterrows():
    top.add_trace(go.Scatter(
        x=month_labels,
        y=row[1:-1],  # skip 'Destination' and 'Total'
        mode='lines+markers',
        name=row['Destination']
    ))

# Step 4: Update the layout
top.update_layout(
    title='Top 10 Corn Destinations by Month',
    xaxis_title='Month',
    yaxis_title='Quantity (mt)',
    template='plotly_white',
    height=600,
    width=950,
    hovermode='x unified'
)

top.show()
top_html = top.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
# Function to format numbers
def format_kmt(value):
    try:
        num = float(str(value).replace(',', ''))
        return f"{num / 1000:.1f}k"
    except:
        return value  # in case it's non-numeric


# Apply formatting to all numeric columns
for col in top_10_countries.columns[1:]:  # Skip 'Destination'
    top_10_countries[col] = top_10_countries[col].apply(format_kmt)

top_10_countries.reset_index(inplace=True)
top_10_countries = top_10_countries.drop(columns='index')
top_10_countries_html=top_10_countries.to_html()


## COORDINATORS

In [0]:
# If January → use second year of the season, else → use first year
if selected_month == 1:
    current_year = end_year
else:
    current_year = start_year

database['Year']=database['Date'].dt.year
database['Coordinator'] = (
    database['Coordinator']
    .astype(str)                # ensure string
    .str.strip()                # remove leading/trailing spaces
    .str.replace(r'\s+', ' ', regex=True)  # replace multiple spaces with single
    .str.title()                # optional: unify capitalization (e.g., "john doe" -> "John Doe")
)

# Filter for current season
df_current_year = database[database['Year'] == current_year]

# Extract marketing month (Dec to Nov)
df_current_year['Marketing_Month'] = df_current_year['Date'].dt.month

# Map month numbers to names, Dec to Nov
month_order = [1,2,3, 4, 5, 6, 7, 8, 9, 10, 11,12]
month_names = ['Jan', 'Feb','Mar', 'Apr', 'May', 'Jun','Jul', 'Aug', 'Sep', 'Oct', 'Nov','Dec']
month_map = dict(zip(month_order, month_names))

df_current_year['Rank'] = df_current_year['Marketing_Month'].map(month_map)

# Replace missing Coordinators with 'UNKNOWN' if needed
df_current_year['Coordinator'] = df_current_year.get('Coordinator', pd.Series()).fillna('UNKNOWN')


# Group by Coordinator and Rank and sum Quantity
pivot_df_current_year_coord = df_current_year.groupby(['Coordinator', 'Rank'])['Quantity'].sum().reset_index()

# Pivot to get desired format
pivot_table_coord = pivot_df_current_year_coord.pivot(index='Coordinator', columns='Rank', values='Quantity')

# Reorder columns to match Dec–Nov sequence
pivot_table_coord = pivot_table_coord.reindex(columns=month_names, fill_value=0)

# Rename index
pivot_table_coord.index.name = 'Coordinators'

# Optional: reset index if you want Coordinators as a column
pivot_table_coord = pivot_table_coord.reset_index()


# Add Total column (row-wise sum)
pivot_table_coord['Total'] = pivot_table_coord[month_names].sum(axis=1)


# Sort by Total descending and keep only top 20
top_20_coord = pivot_table_coord.sort_values(by='Total', ascending=False).head(20)

top_20_coord.insert(1, 'Rank', range(1, len(top_20_coord) + 1))

top_20_coord_html=top_20_coord.to_html()


coord_percent_df = top_20_coord.copy()
month_only = month_names  

# 2. Convertir les colonnes en valeurs numériques si elles sont formatées (optionnel)
# Si nécessaire, décommenter :
# for col in month_only:
#     coord_percent_df[col] = coord_percent_df[col].replace({',': ''}, regex=True).astype(float)

# 3. Calculer les totaux par mois (colonne)
monthly_totals = coord_percent_df[month_only].replace(',', '', regex=True).astype(float).sum(axis=0)

# 4. Calculer les pourcentages
percentage_table_coord = coord_percent_df.copy()
for col in month_only:
    percentage_table_coord[col] = coord_percent_df[col].replace(',', '', regex=True).astype(float) / monthly_totals[col] * 100

# 5. Formater les pourcentages
percentage_table_coord[month_only] = percentage_table_coord[month_only].applymap(lambda x: f"{x:.1f}%" if pd.notnull(x) else "0.0%")

# 6. Compute and format % for the 'Total' column
total_volume = coord_percent_df['Total'].replace(',', '', regex=True).astype(float).sum()

percentage_table_coord['Total'] = coord_percent_df['Total'].replace(',', '', regex=True).astype(float) / total_volume * 100
percentage_table_coord['Total'] = percentage_table_coord['Total'].apply(lambda x: f"{x:.1f}%" if pd.notnull(x) else "0.0%")


percentage_table_coord_html = percentage_table_coord.to_html(index=False)

# Repartir des deux tables : `top_20_shipper` (quantités formatées) et `percentage_table` (pourcentages formatés)

# On crée une copie pour éviter toute modification accidentelle
combined_table_coord = top_20_coord.copy()

month_only = month_names+ ['Total']  # ['Dec', 'Jan', ..., 'Nov']

def format_kmt(value):
    try:
        if pd.isna(value) or str(value).strip() == '':
            return "0.0k"
        num = float(str(value).replace(',', ''))
        return f"{num / 1000:.1f}k"
    except:
        return "0.0k"

# Appliquer le format kmt aux colonnes mensuelles et combiner avec les pourcentages
# Appliquer le format kmt et combiner avec les pourcentages
for col in month_only:
    formatted_qty = top_20_coord[col].apply(format_kmt)
    formatted_pct = percentage_table_coord[col].fillna("0.0%")
    combined_table_coord[col] = formatted_qty + '  ' + formatted_pct


for col in month_only:
    combined_table_coord[col] = combined_table_coord[col].replace(['0.0k  0.0%', 'nank  0.0%', '0.0k (nan%)'], '')


combined_table_coord.reset_index(inplace=False)

combined_table_coord_html=combined_table_coord.to_html(index=False)



In [0]:
# Add year column
database['Year'] = database['Date'].dt.year

# Clean Coordinator column
database['Coordinator'] = (
    database['Coordinator']
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)

# Filter for current year
df_current_year = database[database['Year'] == current_year]

# Extract marketing month
df_current_year['Marketing_Month'] = df_current_year['Date'].dt.month

# Month order & names
month_order = [1,2,3,4,5,6,7,8,9,10,11,12]
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_map = dict(zip(month_order, month_names))

df_current_year['Rank'] = df_current_year['Marketing_Month'].map(month_map)

# Replace missing Coordinators
df_current_year['Coordinator'] = df_current_year['Coordinator'].fillna('UNKNOWN')

# Group and pivot
pivot_df_current_year_coord = (
    df_current_year.groupby(['Coordinator', 'Rank'])['Quantity'].sum().reset_index()
)

pivot_table_coord = pivot_df_current_year_coord.pivot(
    index='Coordinator', columns='Rank', values='Quantity'
).reindex(columns=month_names, fill_value=0)

pivot_table_coord.index.name = 'Coordinators'
pivot_table_coord = pivot_table_coord.reset_index()

# Add totals
pivot_table_coord['Total'] = pivot_table_coord[month_names].sum(axis=1)

# === TOP 20 by total ===
top_20_coord = (
    pivot_table_coord.sort_values(by='Total', ascending=False).head(20).copy()
)
top_20_coord.insert(1, 'Rank', range(1, len(top_20_coord) + 1))

# === PERCENTAGES (relative to whole dataset) ===
monthly_totals = pivot_table_coord[month_names].sum(axis=0)
total_volume = pivot_table_coord['Total'].sum()

percentage_table_coord = top_20_coord.copy()
for col in month_names:
    percentage_table_coord[col] = (
        top_20_coord[col] / monthly_totals[col] * 100
    )

percentage_table_coord['Total'] = (
    top_20_coord['Total'] / total_volume * 100
)

# Format percentages
percentage_table_coord[month_names + ['Total']] = percentage_table_coord[month_names + ['Total']].applymap(
    lambda x: f"{x:.1f}%" if pd.notnull(x) else "0.0%"
)

# === COMBINE FORMATTED QUANTITIES + PERCENTAGES ===
def format_kmt(value):
    try:
        if pd.isna(value) or str(value).strip() == '':
            return "0.0k"
        num = float(str(value).replace(',', ''))
        return f"{num/1000:.1f}k"
    except:
        return "0.0k"

combined_table_coord = top_20_coord.copy()
month_only = month_names + ['Total']

for col in month_only:
    formatted_qty = top_20_coord[col].apply(format_kmt)
    formatted_pct = percentage_table_coord[col].fillna("0.0%")
    combined_table_coord[col] = formatted_qty + '  ' + formatted_pct

# Clean up zeros
for col in month_only:
    combined_table_coord[col] = combined_table_coord[col].replace(
        ['0.0k  0.0%', 'nank  0.0%', '0.0k (nan%)'], ''
    )

# === FINAL HTML TABLE ===
combined_table_coord_html = combined_table_coord.to_html(index=False)


In [0]:
filtered_df['Coordinator'] = filtered_df.get('Coordinator', pd.Series()).fillna('UNKNOWN')

filtered_df['Coordinator'] = (
    filtered_df['Coordinator']
    .astype(str)                # ensure string
    .str.strip()                # remove leading/trailing spaces
    .str.replace(r'\s+', ' ', regex=True)  # replace multiple spaces with single
    .str.title()                # optional: unify capitalization (e.g., "john doe" -> "John Doe")
)
filtered_df['Coordinator'] = filtered_df['Coordinator'].astype(str).str.upper()
grouped_by_coord = filtered_df.groupby('Coordinator')['Quantity'].sum().reset_index()

total_sum_coord = grouped_by_coord['Quantity'].sum()

grouped_by_coord['Percentage'] = (grouped_by_coord['Quantity'] / total_sum_coord) * 100
top_10_coords = grouped_by_coord.sort_values('Percentage', ascending=False).head(10)


fig_coords = go.Figure()

fig_coords.add_trace(go.Pie(
    labels=top_10_coords['Coordinator'],
    values=top_10_coords['Percentage'],
    textinfo='label+percent',
    texttemplate='%{label}<br>%{value:.2f}%',
    insidetextorientation='radial',
    marker=dict(colors=px.colors.sequential.Blues),
    hovertemplate='%{label}<br>%{value:.2f}%<extra></extra>'
))


fig_coords.update_layout(
    title="Top 10 Coordinators",
    template="plotly_white"
)

fig_coords.show()

fig_coords_html = fig_coords.to_html(include_plotlyjs='cdn', full_html=True)


### PORTS

In [0]:
df_current_season=database[database['Season'] == current_season]


In [0]:
# Make sure Marketing_Month_Name exists
df_current_season['Marketing_Month'] = df_current_season['Date'].dt.month

df_current_season['Marketing_Month_Name'] = df_current_season['Marketing_Month'].map(month_map)

# Group by Month and zone
grouped = df_current_season.groupby(['Marketing_Month_Name', 'Port'])['Quantity'].sum().reset_index()

# Ensure month order is preserved
grouped['Marketing_Month_Name'] = pd.Categorical(grouped['Marketing_Month_Name'], categories=month_names, ordered=True)
grouped = grouped.sort_values(by='Marketing_Month_Name')

# Get current month name in marketing year
current_month_number = datetime.now().month
current_marketing_month = month_map[current_month_number]

# Calculate monthly total for % share
monthly_totals = grouped.groupby('Marketing_Month_Name')['Quantity'].transform('sum')
grouped['Pct'] = (grouped['Quantity'] / monthly_totals) * 100


In [0]:
all_months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
start = 'Feb'
i = all_months.index(start)
month_order = all_months[i:] + all_months[:i]  # ['Feb','Mar',...,'Jan']

# Ensure ordered categorical so sorting & axis work
df_current_season['Marketing_Month_Name'] = pd.Categorical(df_current_season['Marketing_Month_Name'],
                                            categories=month_order, ordered=True)
df_current_season = df_current_season.sort_values('Marketing_Month_Name')

# % share per month
monthly_totals = df_current_season.groupby('Marketing_Month_Name')['Quantity'].transform('sum')
df_current_season['Pct'] = (df_current_season['Quantity'] / monthly_totals) * 100

def format_label(x, pct):
    return f"{int(x/1000):,} kmt<br>({pct:.0f}%)"

fig = go.Figure()
ports = df_current_season['Port'].unique()
palette = px.colors.qualitative.Set2

for i, port in enumerate(ports):
    d = df_current_season[df_current_season['Port'] == port]
    fig.add_trace(go.Bar(
        x=d['Marketing_Month_Name'],
        y=d['Quantity'],
        name=port,
        marker_color=palette[i % len(palette)],
        text=[format_label(q, p) for q, p in zip(d['Quantity'], d['Pct'])],
        textposition='outside',
        customdata=d[['Port']],  # attach Port as customdata
        hovertemplate="<b>%{customdata[0]}</b><br>Month: %{x}<br>Quantity: %{y:,.0f} mt<extra></extra>"
    ))
    

fig.update_layout(
    barmode='group',
    title='BRA Exports by Port',
    xaxis_title='Month (marketing year)',
    yaxis_title='Quantity (tons)',
    template='plotly_white',
    legend_title='Port',
    height=700,
    width=950,
    hovermode='x unified'
)

# Force Plotly to use our month order on the x-axis
fig.update_xaxes(categoryorder='array', categoryarray=month_order)

fig.show()
fig_html = fig.to_html(include_plotlyjs='cdn', full_html=True)

In [0]:
import calendar

In [0]:
current_month = calendar.month_name[selected_month]

# Next month handling year wrap (e.g., December to January)
next_month_date = selected_date.replace(day=28) + timedelta(days=4)  # Always lands in next month
next_month = calendar.month_name[next_month_date.month]

In [0]:
corn_html= f"""
<!DOCTYPE html>
<html>
<head>
    <title>BRAZIL CORN LINE UP REPORT</title>
    <style>
body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}

        h1, h2, h3 {{
            color: #2c3e50;
        }}

        table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 10px;
            margin-bottom: 30px;
            font-size: 14px;
        }}

        th, td {{
            border: 1px solid #ccc;
            padding: 8px 12px;
            text-align: left;
        }}

        th {{
            background-color: #f2f2f2;
            color: #333;
            font-weight: bold;
        }}

        tr:nth-child(even) {{
            background-color: #f9f9f9;
        }}

        tr:hover {{
            background-color: #f1f1f1;
        }}

        .layout {{
            width: 100%;
        }}

        .table-container, .extra-container {{
            vertical-align: top;
            padding: 10px;
        }}

        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
      <h1>{current_month} {selected_season} Line up</h1>
     
        <h2>Exports Overview</h2>
            <div class="chart-container">{overview_html}</div>
        <h2>Current week numbers</h2>
            {summary_df_html}  <!-- Insert DataFrame as an HTML table -->
             
      <h1>{current_month} Exports by Destinations</h1>
      <table class="layout">
          <tr>
              <!-- New Container Section (before the chart) -->
              <td class="extra-container">
                  <h2>Destinations</h2>
                  <div class="chart-container">{bar_html}</div>
              </td>
              <!-- Table Section -->
              <td class="table-container">
                <h2>{current_month} Line Ups</h2>
                  {merged_with_subtotals_html}  <!-- Insert DataFrame as an HTML table -->
              </td>
              <!-- Table Section -->
              <td class="table-container">
                <h2>{next_month} Line Ups</h2>
                  {merged_with_subtotals_html_next}  <!-- Insert DataFrame as an HTML table -->
              </td>
              
          </tr>
      </table>
      <h1>{current_season} Top 10 Destinations</h1>
      <table class="layout">
          <tr>
              <!-- New Container Section (before the chart) -->
              <td class="extra-container">
                  <h2>Season {current_season} Main Destinations</h2>
                  <div class="chart-container">{top_html}</div>
              </td>
              <!-- Table Section -->
              <td class="table-container">
                  <h2>Accumulated 2025 exports of main destinations</h2>
                  {top_10_countries_html}  <!-- Insert DataFrame as an HTML table -->
              </td>
          </tr>
      </table>
      <h1>Top 10 Coordinators</h2>
        <div class="chart-container">{fig_coords_html}</div>
        {combined_table_coord_html}

      <h1>Top Ports</h1>

        <div class="chart-container">{fig_html}</div>

  </body>
  </html>
  
  """

corn_report_bytes = corn_html.encode("utf-8")

In [0]:
html_content = f"""
  <!DOCTYPE html>
  <html>
  <head>
      <title>BRAZIL CORN Line up Overview </title>
      <style>
          .container {{
              justify-content: space-between;
              align-items: flex-start;
              width: 100%;
              gap: 10px;
          }}
          .chart-container, .table-container, .extra-container {{
              flex: 0;
              padding: 10px;
              height: 100%;
          }}
          .table-container {{
              text-align: center;
              margin: auto;
              flex: 1; 
              min-width: 300px; 
          }}
          table {{
              width: 60%;
              border-collapse: collapse;
          }}
          th, td {{
              border: 1px solid black;
              padding: 8px;
              text-align: center;
          }}
          th {{
              background-color: #f2f2f2;
          }}
          img {{
              width: 100%;
              height: auto;
              min-width: 750px; /* Set a minimum width */
              min-height: 750px; 
              object-fit: contain;
          }}
          .wide-table {{
              width: 100%;
              table-layout: fixed;
          }}

          .wide-table th, .wide-table td {{
              padding: 8px;
              width: 40%;
              min-width: 100px;
              white-space: nowrap;
          }}
          
      </style>
  </head>
  <body>
      <h1>BRAZIL CORN {current_month} Exports</h1>
     
        <h2>Exports Overview</h2>
            <img src="chart1" alt="Corn Exports overview">
        <h2>Current week numbers</h2>
            {summary_df_html}  <!-- Insert DataFrame as an HTML table -->
             
      <h1>{current_month} Exports by Destinations</h1>
      <table class="layout">
          <tr>
              <!-- New Container Section (before the chart) -->
              <td class="extra-container">
                  <h2>Destinations</h2>
                  <img src="chart2" alt="Board Crush Chart">
              </td>
              <!-- Table Section -->
              <td class="table-container">
                <h2>{current_month} Line Ups</h2>
                  {merged_with_subtotals_html}  <!-- Insert DataFrame as an HTML table -->
              </td>
              <!-- Table Section -->
              <td class="table-container">
                <h2>{next_month} Line Ups</h2>
                  {merged_with_subtotals_html_next}  <!-- Insert DataFrame as an HTML table -->
              </td>
              
          </tr>
      </table>
      <h1>{current_season} Top 10 Destinations</h1>
      <table class="layout">
          <tr>
              <!-- New Container Section (before the chart) -->
              <td class="extra-container">
                  <h2>Season {current_season} Main Destinations</h2>
                  <img src="chart3" alt="top 10">
              </td>
              <!-- Table Section -->
              <td class="table-container">
                  <h2>Accumulated 2025 exports of main destinations</h2>
                  {top_10_countries_html}  <!-- Insert DataFrame as an HTML table -->
              </td>
          </tr>
      </table>
        <h1>Top 10 Coordinators</h2>
        <img src="chart5" alt="Top Coordinators">
        {combined_table_coord_html}

      <h1>Top Ports</h1>

        <img src="chart6" alt="Top ports">

  </body>
  </html>
  """


# # Send email with embedded chart and table
LDCDataAccessLayerPy.mail.mail_send(
    to=email_to_send,
    subject=f'BRAZIL Corn Line up {selected_date.strftime("%d-%m")}',
    from_addr="florian.girardi-ext@ldc.com",
    body=html_content,
    mime_type="html",
    html_images={"chart1": overview,"chart2":bar,'chart3':top,"chart5":fig_coords,'chart6':fig},
    attachment={'bra_corn_lineup.html':corn_report_bytes}   # Embedding the chart image
)